# Phase-SNN Intent Classifier v5 — Fixed Optimizers + Best Model

**Runtime → T4 GPU → Run All (~15 min)**

Fixes from v4:
- Separate Adam optimizers for CE and geometric losses (no more corrupted moments)
- Best model checkpoint saved and restored before final eval
- 600 epoch training to find true convergence ceiling

| Model | Expected test acc | Size |
|-------|-----------------|------|
| GloVe mean-pool+cosine | 56.8% | 59 KB |
| Phase+CE K=270 v4 (broken) | 50.8% | 265 KB |
| Phase+CE K=270 v5 (fixed) | ~78-82%? | 265 KB |
| DistilBERT | 95.1% | 66 MB |


## Cell 1: Install phase_snn_v2

In [ ]:
import base64, os, sys
sys.path.insert(0, '/content')
_V2_B64 = '...'
with open('/content/phase_snn_v2.py', 'w') as fh:
    fh.write(base64.b64decode(_V2_B64).decode())
import phase_snn_v2 as v2
print(f'phase_snn_v2 installed: {os.path.getsize("/content/phase_snn_v2.py"):,} bytes')


## Cell 2: Full Pipeline
Runs all steps. ~15 min on T4.

In [ ]:
"""
Phase-SNN Intent Classifier
============================
Minimal intent classification pipeline.
Input: raw text
Output: intent class

Three variants compared on identical data:
  A. Random baseline (majority class)
  B. GloVe mean-pool + cosine (no phase encoding)
  C. Phase encoder + cosine (our architecture)

Target dataset: CLINC150
Target metric: accuracy vs DistilBERT (95.1%) at <1MB model size
"""

import os, sys, time, json, gc
import numpy as np
from scipy.special import expit as sigmoid
from collections import defaultdict

# ── 0. Install & imports ──────────────────────────────────────────────────────
import subprocess
def install(pkg):
    subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'], check=True)

install('datasets')
install('nltk')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import word_tokenize

sys.path.insert(0, '/content')
import phase_snn_v2 as v2

# ── 1. Load CLINC150 ──────────────────────────────────────────────────────────
print("="*60)
print("STEP 1: LOAD CLINC150")
print("="*60)

from datasets import load_dataset
dataset = load_dataset('clinc_oos', 'plus')

# Extract splits
def extract_split(split):
    texts   = [x['text']   for x in split]
    intents = [x['intent'] for x in split]
    return texts, intents

train_texts, train_intents = extract_split(dataset['train'])
val_texts,   val_intents   = extract_split(dataset['validation'])
test_texts,  test_intents  = extract_split(dataset['test'])

# Get label names from dataset features (labels are integers in CLINC150)
label_feature = dataset['train'].features['intent']
idx_to_intent = {i: label_feature.int2str(i) for i in range(label_feature.num_classes)}
intent_to_idx = {v: k for k, v in idx_to_intent.items()}
N_INTENTS     = label_feature.num_classes
all_intents   = list(idx_to_intent.values())  # string names

# Labels are already integers
train_labels = np.array(train_intents)
val_labels   = np.array(val_intents)
test_labels  = np.array(test_intents)

print(f"Train: {len(train_texts):,} examples")
print(f"Val:   {len(val_texts):,} examples")
print(f"Test:  {len(test_texts):,} examples")
print(f"Intents: {N_INTENTS}")
print(f"Sample labels: {[idx_to_intent[i] for i in range(min(5,N_INTENTS))]}")

# Check for OOS
oos_label = intent_to_idx.get('oos', None)
print(f"OOS class index: {oos_label}")

# ── 2. Load GloVe ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 2: LOAD GLOVE")
print("="*60)

GLOVE_PATH = '/content/glove.6B.100d.txt'
if not os.path.exists(GLOVE_PATH):
    print("Downloading GloVe 100d...")
    subprocess.run(['wget','-q','-O','/content/glove.6B.zip',
        'https://nlp.stanford.edu/data/glove.6B.zip'], check=True)
    subprocess.run(['unzip','-q','/content/glove.6B.zip',
        'glove.6B.100d.txt','-d','/content/'], check=True)

print("Loading GloVe...")
glove = {}
with open(GLOVE_PATH, encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        glove[parts[0]] = np.array(parts[1:], dtype=np.float32)
D = 100
print(f"Loaded {len(glove):,} vectors  D={D}")

# ── 3. Text → Embedding ───────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 3: BUILD SENTENCE EMBEDDINGS")
print("="*60)

def text_to_embedding(text, glove, D):
    """Tokenize, GloVe lookup, mean pool. Returns (D,) unit vector."""
    tokens = word_tokenize(text.lower())
    vecs = [glove[t] for t in tokens if t in glove]
    if not vecs:
        return np.zeros(D, dtype=np.float32)
    v = np.mean(vecs, axis=0)
    norm = np.linalg.norm(v)
    return (v / norm).astype(np.float32) if norm > 1e-12 else v

def build_embeddings(texts, glove, D, desc=""):
    embs = np.array([text_to_embedding(t, glove, D) for t in texts],
                    dtype=np.float32)
    coverage = np.mean(np.linalg.norm(embs, axis=1) > 1e-6)
    print(f"  {desc}: {len(embs):,} embeddings  coverage={coverage:.1%}")
    return embs

t0 = time.time()
train_embs = build_embeddings(train_texts, glove, D, "train")
val_embs   = build_embeddings(val_texts,   glove, D, "val")
test_embs  = build_embeddings(test_texts,  glove, D, "test")
print(f"  Embedding time: {time.time()-t0:.1f}s")

# ── 4. Baseline A: Majority class ─────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 4: BASELINE A — MAJORITY CLASS")
print("="*60)

from collections import Counter
majority = Counter(train_labels.tolist()).most_common(1)[0][0]
maj_acc_test = np.mean(test_labels == majority)
print(f"Majority class: {idx_to_intent[majority]}")
print(f"Test accuracy:  {maj_acc_test:.4f}")

# ── 5. Baseline B: GloVe mean-pool + cosine ───────────────────────────────────
print("\n" + "="*60)
print("STEP 5: BASELINE B — GLOVE MEAN-POOL + COSINE")
print("="*60)

def build_prototypes(embs, labels, n_classes):
    """Mean embedding per class, L2-normalised."""
    protos = np.zeros((n_classes, embs.shape[1]), dtype=np.float32)
    for c in range(n_classes):
        mask = labels == c
        if mask.sum() > 0:
            p = embs[mask].mean(axis=0)
            norm = np.linalg.norm(p)
            protos[c] = p / norm if norm > 1e-12 else p
    return protos

def cosine_classify(embs, prototypes):
    """Nearest prototype by cosine similarity."""
    # Normalise query embeddings
    norms = np.linalg.norm(embs, axis=1, keepdims=True) + 1e-12
    embs_n = embs / norms
    scores = embs_n @ prototypes.T   # (N, C)
    return np.argmax(scores, axis=1)

def evaluate(preds, labels, split_name, oos_label=None):
    # Overall accuracy
    acc = np.mean(preds == labels)
    # In-scope accuracy (exclude OOS)
    if oos_label is not None:
        mask = labels != oos_label
        acc_inscope = np.mean(preds[mask] == labels[mask])
    else:
        acc_inscope = acc
    print(f"  {split_name}: acc={acc:.4f}  in-scope={acc_inscope:.4f}")
    return acc, acc_inscope

t0 = time.time()
glove_protos = build_prototypes(train_embs, train_labels, N_INTENTS)
glove_preds_val  = cosine_classify(val_embs,  glove_protos)
glove_preds_test = cosine_classify(test_embs, glove_protos)
glove_time = (time.time() - t0) / len(test_texts) * 1000

print(f"GloVe prototype size: {glove_protos.nbytes:,} bytes")
val_acc_glove,  _ = evaluate(glove_preds_val,  val_labels,  "val",  oos_label)
test_acc_glove, _ = evaluate(glove_preds_test, test_labels, "test", oos_label)
print(f"Inference: {glove_time:.3f}ms/query")

# ── 6. Build relation pairs for phase training ────────────────────────────────
print("\n" + "="*60)
print("STEP 6: BUILD RELATIONAL TRAINING STRUCTURE")
print("="*60)

# Use the 10 official CLINC150 domains (from the paper).
# Each domain has ~15 intents — clean, meaningful relational families.
CLINC150_DOMAINS = {
    'banking':           ['transfer','transactions','balance','freeze_account',
                          'pay_bill','bill_balance','bill_due','interest_rate',
                          'routing','min_payment','new_card','lost_card',
                          'card_decline','pin_change','report_fraud'],
    'credit_cards':      ['credit_score','report_lost_card','credit_limit',
                          'rewards_balance','application_status',
                          'card_about_to_expire','replacement_card_duration',
                          'expiration_date','credit_limit_change','damaged_card',
                          'improve_credit_score','apr','redeem_rewards',
                          'account_blocked','spending_history'],
    'kitchen_and_dining':['recipe','food_last','meal_suggestion','nutrition_info',
                          'calories','ingredient_substitution','cook_time',
                          'food_beverage_price','restaurant_reviews',
                          'restaurant_reservation','confirm_reservation',
                          'how_busy','cancel_reservation','accept_reservations',
                          'ingredients_list'],
    'home':              ['smart_home','shopping_list','shopping_list_update',
                          'next_song','play_music','update_playlist','todo_list',
                          'todo_list_update','calendar','calendar_update',
                          'order','order_status','reminder','reminder_update',
                          'what_can_i_ask'],
    'auto_and_commute':  ['car_rental','car_bluetooth','tire_pressure',
                          'oil_change_when','oil_change_how','jump_start',
                          'uber','schedule_maintenance','last_maintenance',
                          'insurance','traffic','directions','gas','gas_type',
                          'distance'],
    'travel':            ['book_flight','book_hotel','get_hotel_recommendations',
                          'travel_suggestion','travel_notification',
                          'carry_on_baggage','timezone','international_visa',
                          'plug_type','exchange_rate','flight_status',
                          'international_fees','vaccines','lost_luggage','mpg'],
    'utility':           ['time','alarm','timer','weather','date','find_phone',
                          'share_location','current_location',
                          'meeting_schedule','calculator',
                          'measurement_conversion','spelling','definition',
                          'change_accent','sync_device'],
    'work':              ['direct_deposit','pto_request','taxes','payday','w2',
                          'income','rollover_401k','find_internship','fico_score',
                          'insurance_change','user_name','password_reset',
                          'change_user_name','change_password','next_holiday'],
    'meta':              ['who_do_you_work_for','do_you_have_pets','are_you_a_bot',
                          'meaning_of_life','who_made_you','thank_you','goodbye',
                          'tell_joke','where_are_you_from','how_old_are_you',
                          'what_is_your_name','what_are_your_hobbies','fun_fact',
                          'change_ai_name','what_can_i_ask'],
    'small_talk':        ['greeting','yes','no','maybe','I_am_bored',
                          'flip_coin','roll_dice','laugh','story','text',
                          'repeat','whisper_mode','make_call','number_facts',
                          'next_holiday'],
}

# Map intent name -> domain, only for intents present in this dataset
intent_to_domain = {}
for domain, intent_names in CLINC150_DOMAINS.items():
    for name in intent_names:
        if name in intent_to_idx:
            intent_to_domain[name] = domain

covered     = len(intent_to_domain)
total_nooos = N_INTENTS - (1 if oos_label is not None else 0)
print(f"Domain coverage: {covered}/{total_nooos} intents mapped")

# Build domain -> intent index list
domain_to_intents = defaultdict(list)
for intent_name, domain in intent_to_domain.items():
    domain_to_intents[domain].append(intent_to_idx[intent_name])

print(f"\nDomains (R):")
for domain, idxs in sorted(domain_to_intents.items()):
    print(f"  {domain:<25} {len(idxs):>3} intents")

# Build relation pairs: training example pairs from different intents
# within the same domain
rng_pairs = np.random.default_rng(42)
rel_pairs_by_domain = []
for domain, domain_intent_idxs in domain_to_intents.items():
    if len(domain_intent_idxs) < 2:
        continue
    pairs = []
    for ia in range(len(domain_intent_idxs)):
        for ib in range(ia+1, len(domain_intent_idxs)):
            ex_a = np.where(train_labels == domain_intent_idxs[ia])[0]
            ex_b = np.where(train_labels == domain_intent_idxs[ib])[0]
            if len(ex_a) == 0 or len(ex_b) == 0:
                continue
            for i in range(min(3, len(ex_a), len(ex_b))):
                pairs.append((int(ex_a[i]), int(ex_b[i])))
    if len(pairs) >= 5:
        rel_pairs_by_domain.append(pairs)

print(f"\nRelation families with >=5 pairs: {len(rel_pairs_by_domain)}")
for i, p in enumerate(rel_pairs_by_domain):
    print(f"  Family {i}: {len(p):,} pairs")

R = len(rel_pairs_by_domain)
# ── 7. Train Phase Encoder + Classification Head jointly ────────────────────
print("\n" + "="*60)
print("STEP 7: TRAIN PHASE ENCODER + CE CLASSIFICATION HEAD")
print("="*60)

# Architecture: Phase encoder (D -> K) + linear head (K -> N_INTENTS)
# Loss: cross-entropy (discriminative) + metric + relational (geometric)
# CE loss backprops through the phase encoder — joint optimisation.

K = min(320, max(128, int(np.ceil(27 * R / 10) * 10)))
print(f"R={R}  K={K}  K_pos={int(0.7*K)}  K_rel={K-int(0.7*K)}")
print(f"Classification head: ({N_INTENTS}, {K})")
print(f"Total params: {K*D + N_INTENTS*K:,}  "
      f"({(K*D + N_INTENTS*K)*4/1024:.1f} KB)")

sigmoid_fn = sigmoid  # alias for use inside CE function

def ce_forward_and_grads(E_b, labels, W_enc, omega, W_cls, K_dim):
    """CE loss + gradients w.r.t. W_enc and W_cls."""
    B  = len(E_b)
    Ed = E_b.astype(np.float64)
    z   = Ed @ W_enc.T                              # (B, K)
    sig = sigmoid_fn(z)
    phi = 2 * np.pi * sig * omega[None, :]          # (B, K)
    logits = phi @ W_cls.T                          # (B, N_CL)
    ex  = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = ex / ex.sum(axis=1, keepdims=True)
    loss  = -np.mean(np.log(probs[np.arange(B), labels] + 1e-12))
    delta = probs.copy()
    delta[np.arange(B), labels] -= 1.0
    delta /= B
    grad_W_cls = delta.T @ phi                      # (N_CL, K)
    d_phi = delta @ W_cls                           # (B, K)
    sp    = sig * (1 - sig)
    scale = 2 * np.pi * omega[None, :] * d_phi * sp
    grad_W_enc = scale.T @ Ed                       # (K, D)
    return loss, grad_W_enc, grad_W_cls

EPOCHS     = 300
BATCH_SIZE = 256
LAM_CE     = 1.0    # cross-entropy weight
LAM_METRIC = 0.3    # geometric regularisation
LAM_XFER   = 0.1    # relational regularisation
LAM_REL    = 0.05

enc = v2.PhaseEncoderV2(D, K, sampled=True, batch_size=BATCH_SIZE,
    hard_ratio=0.5, freq_bands=True, train_omega=True, partition=True,
    lam_metric=LAM_METRIC, lam_xfer=LAM_XFER, lam_rel=LAM_REL, lr=2e-3)

np.random.seed(42)
W_cls    = np.random.randn(N_INTENTS, K).astype(np.float64) * 0.01
opt_cls  = v2.Adam((N_INTENTS, K), lr=5e-3)
# Separate Adam for CE encoder gradient — keeps CE and geometric
# moment estimates independent so they don't corrupt each other
opt_enc_ce = v2.Adam((K, D), lr=2e-3)

rng_batch = np.random.default_rng(42)
t0 = time.time()
best_val_acc = 0.0
best_W_enc   = enc.W.copy()
best_W_cls   = W_cls.copy()
best_omega   = enc.omega.copy()
best_ep      = 0

print(f"\nTraining {EPOCHS} epochs  CE={LAM_CE}  metric={LAM_METRIC}"
      f"  xfer={LAM_XFER}  rel={LAM_REL}")

for ep in range(1, EPOCHS + 1):
    # ── CE step ──────────────────────────────────────────────────────────
    idx  = rng_batch.choice(len(train_embs), BATCH_SIZE, replace=False)
    E_b  = train_embs[idx]
    labs = train_labels[idx]
    ce_loss, gW_enc, gW_cls = ce_forward_and_grads(
        E_b, labs, enc.W, enc.omega, W_cls, K)

    # Apply CE gradient via its own optimizer (separate from geometric Adam)
    enc.W -= opt_enc_ce.step(LAM_CE * gW_enc)
    W_cls -= opt_cls.step(LAM_CE * gW_cls)

    # ── Geometric losses (metric + relational) ────────────────────────────
    geo = enc.step(train_embs, None, quads_pos, quads_neg)

    if ep % 100 == 0 or ep == EPOCHS:
        # Quick val accuracy
        phi_val = enc.phi(val_embs.astype(np.float64)).astype(np.float32)
        val_logits = phi_val @ W_cls.T.astype(np.float32)
        val_preds  = np.argmax(val_logits, axis=1)
        val_acc    = np.mean(val_preds == val_labels)
        # Save best model weights
        if val_acc > best_val_acc:
            best_val_acc  = val_acc
            best_W_enc    = enc.W.copy()
            best_W_cls    = W_cls.copy()
            best_omega    = enc.omega.copy()
            best_ep       = ep
        print(f"  ep={ep:>3}  ce={ce_loss:.4f}"
              f"  metric={geo.get('metric',0):.4f}"
              f"  val_acc={val_acc:.4f}"
              f"  {'★ best' if val_acc == best_val_acc else '':6}"
              f"  t={time.time()-t0:.0f}s")

print(f"Training complete: {time.time()-t0:.1f}s")
print(f"Best val_acc={best_val_acc:.4f} at epoch {best_ep} — restoring best weights")
# Restore best weights for final evaluation
enc.W     = best_W_enc
enc.omega = best_omega
W_cls     = best_W_cls

# ── 8. Evaluate ───────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 8: EVALUATE")
print("="*60)

def classify(embs, enc, W_cls):
    phi = enc.phi(embs.astype(np.float64)).astype(np.float32)
    logits = phi @ W_cls.T.astype(np.float32)
    return np.argmax(logits, axis=1), phi

# Inference time
t_inf = time.time()
for _ in range(100):
    classify(test_embs[:1], enc, W_cls)
inf_ms = (time.time() - t_inf) / 100 * 1000

val_preds,  val_phi  = classify(val_embs,  enc, W_cls)
test_preds, test_phi = classify(test_embs, enc, W_cls)

def acc_report(preds, labels, split, oos_lbl):
    acc = np.mean(preds == labels)
    if oos_lbl is not None:
        mask = labels != oos_lbl
        ins  = np.mean(preds[mask] == labels[mask])
    else:
        ins = acc
    print(f"  {split}: acc={acc:.4f}  in-scope={ins:.4f}")
    return acc, ins

val_acc,  val_ins  = acc_report(val_preds,  val_labels,  "val",  oos_label)
test_acc, test_ins = acc_report(test_preds, test_labels, "test", oos_label)
print(f"  Inference: {inf_ms:.3f}ms/query")

model_size = (K * D + N_INTENTS * K) * 4   # float32 bytes

# ── 9. FINAL REPORT ───────────────────────────────────────────────────────────
print("\n\n" + "="*68)
print("INTENT CLASSIFICATION — FINAL REPORT")
print("Dataset: CLINC150 (plus split, includes OOS)")
print("="*68)

model_size_glove = N_INTENTS * D * 4
# model_size_phase now computed in Step 7

print(f"\n  {'Model':<30}  {'val acc':>8}  {'test acc':>9}  "
      f"{'size':>10}  {'ms/q':>6}")
print(f"  {'-'*68}")

rows = [
    ("Majority class",           "—",
                                 f"{maj_acc_test:.4f}",   "0 bytes",      "~0"),
    ("GloVe mean-pool+cosine",   f"{val_acc_glove:.4f}",
                                 f"{test_acc_glove:.4f}",
                                 f"{N_INTENTS*D*4//1024} KB",
                                 f"{glove_time:.2f}"),
    (f"Phase+CE K={K} (ours)",   f"{val_acc:.4f}",
                                 f"{test_acc:.4f}",
                                 f"{model_size//1024} KB",
                                 f"{inf_ms:.2f}"),
    ("DistilBERT (published)",   "—",   "0.9510",
                                 "66,000 KB",              "~10-50"),
]
for label, val_a, test_a, size, ms in rows:
    print(f"  {label:<30}  {val_a:>8}  {test_a:>9}  {size:>10}  {ms:>6}")

print(f"\n  Phase+CE model size: {model_size:,} bytes ({model_size/1024:.1f} KB)")
print(f"  DistilBERT:          66,000,000 bytes (66,000 KB)")
print(f"  Size ratio:          {model_size/66e6*100:.2f}% of DistilBERT")

delta_glove = test_acc - test_acc_glove
delta_dist  = 0.9510 - test_acc
print(f"\n  Phase+CE vs GloVe baseline:  {delta_glove:+.4f}")
print(f"  Gap to DistilBERT:           {delta_dist:.4f}")

if test_acc > test_acc_glove:
    print(f"  ✓ Phase+CE IMPROVES over raw GloVe — CE+geometric jointly helping")
else:
    print(f"  ✗ Phase+CE below GloVe — CE alone not enough, architecture mismatch")

if delta_dist < 0.10:
    print(f"  ✓ Within 10pts of DistilBERT — THESIS CLAIM VIABLE")
elif delta_dist < 0.20:
    print(f"  ~ Within 20pts — promising, continue")
else:
    print(f"  ✗ Gap > 20pts — needs more work")

print("\nDone.")


## Cell 3: Recovery
If notebook crashes, run this then re-run Cell 2.

In [ ]:
import importlib, sys, os
sys.path.insert(0, '/content')
if os.path.exists('/content/phase_snn_v2.py'):
    import phase_snn_v2 as v2; importlib.reload(v2)
    print('Recovery OK.')
else:
    print('Re-run Cell 1 first.')
